### 📦 Offline Dependency Installation

- Installs `datasets` into a local directory (`/kaggle/working/packages`)
- Reuses Kaggle offline packages if available
- Keeps the baseline notebook style so the runtime setup stays familiar

✅ Kaggle-compatible setup for the Phase 1 assistant-only notebook

In [ ]:
import subprocess, sys, os
from pathlib import Path

def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read().strip()
            rel_pack_path = (pth_file.parent / relpath)
            if rel_pack_path.exists():
                print(f"append {rel_pack_path}")
                sys.path.append(str(rel_pack_path))

offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
target_dir = "/kaggle/working/packages"
os.makedirs(target_dir, exist_ok=True)
resolve_python_path(Path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/"))

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index",
        "--find-links", offline_dir,
        "--target", target_dir,
        "datasets"
    ])
    print("Installed datasets from offline packages")

sys.path.append(target_dir)
resolve_python_path(Path(target_dir))


### ⚙️ Environment Setup & Libraries

- Keeps the baseline notebook environment style
- Switches the training stack from `TRL SFTTrainer` to `transformers.Trainer`
- Uses assistant-only labels instead of full-text loss

✅ Phase 1 plumbing aligned with the new plan

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys, stat, shutil, gc, zipfile, csv, json, random
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable

import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType


### 🛠️ Triton + RMSNorm Fix (Kaggle Hack)

- Preserves the baseline notebook's Kaggle-specific Triton patch
- Keeps the runtime behavior close to the original notebook

✅ Minimizes unrelated changes outside the Phase 1 training logic

In [ ]:
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton ptxas fix applied.")


### 📊 Phase 1 Config + Kaggle Paths

**Goal:**
- Reimplement the Phase 1 assistant-only training path as a Kaggle notebook
- Keep the original baseline hyperparameters and LoRA settings where possible

**Data inputs:**
- Preferred: prebuilt Phase 1 CSV from a Kaggle dataset
- Fallback: rebuild the CSV locally from `train_row_analysis_v1.csv`

✅ Uses Kaggle-style paths instead of local repo paths

In [ ]:
# === Hyperparameters ===
SUBSAMPLE_SIZE = 900
LORA_RANK = 32
MAX_SEQ_LEN = 2048
NUM_EPOCHS = 2
BATCH_SIZE = 1
GRAD_ACCUM = 4
LR = 1e-4
SEED = 42
BOXED_INSTRUCTION = "Please put your final answer inside `\boxed{}`. For example: `\boxed{your answer}`"
OUTPUT_DIR = "/kaggle/working/adapter_phase1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Kaggle paths ===
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
PREBUILT_PHASE1_DATASET_CSV = "/kaggle/input/datasets/renta0426/phase1-assistant-only-training-data/phase1_assistant_only_training_data.csv"
SOURCE_ANALYSIS_CSV = "/kaggle/input/datasets/renta0426/nemotron-train-row-analysis-v1/train_row_analysis_v1.csv"
WORKING_DATASET_CSV = "/kaggle/working/phase1_assistant_only_training_data.csv"
WORKING_MANIFEST_JSON = "/kaggle/working/phase1_assistant_only_manifest.json"

VERIFIED_QUOTAS = {
    "gravity_constant": 120,
    "unit_conversion": 120,
    "roman_numeral": 120,
    "text_decryption": 100,
    "bit_manipulation": 160,
    "symbol_equation": 60,
}
ANSWER_ONLY_QUOTAS = {
    "text_decryption": 20,
    "bit_manipulation": 120,
    "symbol_equation": 80,
}
FAMILY_LABELS = {
    "gravity_constant": "gravity",
    "unit_conversion": "unit",
    "roman_numeral": "roman",
    "text_decryption": "text",
    "bit_manipulation": "binary",
    "symbol_equation": "symbol",
}
OUTPUT_COLUMNS = [
    "id", "prompt", "answer", "generated_cot", "label", "assistant_style", "source_selection_tier"
]
REQUIRED_SOURCE_COLUMNS = {
    "id", "prompt", "answer", "family", "template_subtype", "teacher_solver_candidate", "selection_tier", "family_analysis_json"
}


### 🧱 Phase 1 Dataset Builder

- Keeps the Phase 1 script logic inside the notebook
- Builds a 900-row mix of `verified_trace_ready` and `answer_only_keep`
- Uses `boxed_only` for answer-only rows to avoid inventing reasoning

✅ Notebook can either consume a prebuilt CSV or rebuild it on Kaggle

In [ ]:
@dataclass(frozen=True)
class SelectedRow:
    row: dict[str, str]
    rank_key: tuple[Any, ...]


def parse_bool(value: Any) -> bool:
    return str(value).strip().lower() in {"1", "true", "yes"}


def parse_int(value: Any, default: int = 0) -> int:
    try:
        text = str(value).strip()
        if not text:
            return default
        return int(float(text))
    except (TypeError, ValueError):
        return default


def parse_float(value: Any, default: float | None = None) -> float | None:
    try:
        text = str(value).strip()
        if not text:
            return default
        return float(text)
    except (TypeError, ValueError):
        return default


def load_source_rows(path: str) -> list[dict[str, str]]:
    with open(path, "r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames is None:
            raise ValueError(f"CSV header is missing: {path}")
        missing = REQUIRED_SOURCE_COLUMNS.difference(reader.fieldnames)
        if missing:
            raise ValueError(f"Missing required columns in {path}: {sorted(missing)}")
        return [dict(row) for row in reader]


def parse_family_payload(row: dict[str, str]) -> dict[str, Any]:
    text = row.get("family_analysis_json", "")
    if not text:
        return {}
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        return {}
    return payload if isinstance(payload, dict) else {}


def solver_priority(row: dict[str, str]) -> int:
    solver = row.get("teacher_solver_candidate", "")
    priorities = {
        "roman_standard": 100,
        "gravity_numeric_rule": 100,
        "unit_numeric_rule": 100,
        "text_char_substitution": 95,
        "text_word_dictionary": 90,
        "binary_structured_byte_formula": 96,
        "binary_structured_byte_formula_abstract": 93,
        "binary_affine_xor": 92,
        "binary_two_bit_boolean": 91,
        "binary_three_bit_boolean": 89,
        "binary_byte_transform": 88,
        "binary_bit_permutation_bijection": 87,
        "binary_bit_permutation_independent": 85,
        "symbol_numeric_operator_formula": 72,
        "symbol_char_substitution": 68,
    }
    return priorities.get(solver, 50)


def structured_support_strength(row: dict[str, str]) -> int:
    safe_support = parse_int(row.get("bit_structured_formula_safe_support", "0"), 0)
    abstract_support = parse_int(row.get("bit_structured_formula_abstract_support", "0"), 0)
    return max(safe_support * 10, abstract_support)


def rank_verified_row(row: dict[str, str]) -> tuple[Any, ...]:
    example_count = parse_int(row.get("num_examples", "0"), 0)
    same_operator = parse_int(row.get("symbol_same_operator_example_count", "0"), 0)
    hard_score = parse_float(row.get("hard_score", ""), 999.0)
    if hard_score is None:
        hard_score = 999.0
    return (
        -solver_priority(row),
        -example_count,
        -same_operator,
        -structured_support_strength(row),
        hard_score,
        row.get("id", ""),
    )


def rank_answer_only_row(row: dict[str, str]) -> tuple[Any, ...]:
    family = row.get("family", "")
    example_count = parse_int(row.get("num_examples", "0"), 0)
    hard_score = parse_float(row.get("hard_score", ""), 999.0)
    if hard_score is None:
        hard_score = 999.0
    if family == "bit_manipulation":
        return (
            -hard_score,
            -example_count,
            -structured_support_strength(row),
            -solver_priority(row),
            row.get("id", ""),
        )
    return (
        hard_score,
        -example_count,
        -solver_priority(row),
        row.get("id", ""),
    )


def make_selected(items: list[dict[str, str]], ranker: Callable[[dict[str, str]], tuple[Any, ...]]) -> list[SelectedRow]:
    return [SelectedRow(row=item, rank_key=ranker(item)) for item in items]


def round_robin_select(items: list[SelectedRow], quota: int, seed: int, key_fn: Callable[[SelectedRow], str]) -> list[dict[str, str]]:
    groups: dict[str, list[SelectedRow]] = defaultdict(list)
    for item in items:
        groups[key_fn(item)].append(item)
    group_names = sorted(groups)
    rng = random.Random(seed)
    for group_name in group_names:
        groups[group_name].sort(key=lambda item: item.rank_key)
    order = group_names[:]
    rng.shuffle(order)
    queues = {group_name: deque(groups[group_name]) for group_name in order}
    selected: list[dict[str, str]] = []
    while len(selected) < quota and queues:
        for group_name in list(order):
            queue = queues.get(group_name)
            if not queue:
                continue
            selected.append(queue.popleft().row)
            if len(selected) >= quota:
                break
            if not queue:
                queues.pop(group_name, None)
        order = [group_name for group_name in order if group_name in queues]
    if len(selected) != quota:
        available = sum(len(group_items) for group_items in groups.values())
        raise ValueError(f"Unable to satisfy quota {quota}; only {available} rows available")
    return selected


def verified_rows(rows: list[dict[str, str]]) -> list[dict[str, str]]:
    selected: list[dict[str, str]] = []
    for row in rows:
        if row.get("selection_tier") != "verified_trace_ready":
            continue
        if not parse_bool(row.get("verified_trace_ready", "true")):
            continue
        if not row.get("teacher_solver_candidate", ""):
            continue
        selected.append(row)
    return selected


def answer_only_rows(rows: list[dict[str, str]]) -> list[dict[str, str]]:
    return [row for row in rows if row.get("selection_tier") == "answer_only_keep"]


def select_verified_mix(rows: list[dict[str, str]], seed: int) -> list[dict[str, str]]:
    by_family: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in rows:
        family = row.get("family", "")
        if family in VERIFIED_QUOTAS:
            by_family[family].append(row)
    selected: list[dict[str, str]] = []
    for index, family in enumerate(VERIFIED_QUOTAS):
        family_rows = by_family.get(family, [])
        quota = VERIFIED_QUOTAS[family]
        if len(family_rows) < quota:
            raise ValueError(f"Verified family {family} has only {len(family_rows)} rows; needs {quota}")
        family_selected = round_robin_select(
            make_selected(family_rows, rank_verified_row),
            quota=quota,
            seed=seed + index,
            key_fn=lambda item: item.row.get("template_subtype", "unknown"),
        )
        selected.extend(family_selected)
    return selected


def answer_only_group_key(row: dict[str, str]) -> str:
    family = row.get("family", "")
    if family == "bit_manipulation":
        return row.get("teacher_solver_candidate", "") or row.get("template_subtype", "unknown")
    if family == "symbol_equation":
        return row.get("symbol_query_operator", "") or row.get("template_subtype", "unknown")
    return row.get("template_subtype", "unknown")


def select_answer_only_mix(rows: list[dict[str, str]], used_ids: set[str], seed: int) -> list[dict[str, str]]:
    remaining = [row for row in rows if row.get("id", "") not in used_ids]
    by_family: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in remaining:
        family = row.get("family", "")
        if family in ANSWER_ONLY_QUOTAS:
            by_family[family].append(row)
    selected: list[dict[str, str]] = []
    for index, family in enumerate(ANSWER_ONLY_QUOTAS):
        family_rows = by_family.get(family, [])
        quota = ANSWER_ONLY_QUOTAS[family]
        if len(family_rows) < quota:
            raise ValueError(f"Answer-only family {family} has only {len(family_rows)} rows; needs {quota}")
        family_selected = round_robin_select(
            make_selected(family_rows, rank_answer_only_row),
            quota=quota,
            seed=seed + 100 + index,
            key_fn=lambda item: answer_only_group_key(item.row),
        )
        selected.extend(family_selected)
    return selected


In [ ]:
def humanize_rule_name(rule_name: str) -> str:
    return rule_name.replace("_", " ") if rule_name else "verified transformation"


def build_roman_trace(row: dict[str, str], payload: dict[str, Any]) -> str:
    query = row.get("query_raw", "").strip()
    value = parse_int(row.get("roman_query_value", payload.get("query_value", "0")), 0)
    return f"<think>The examples follow standard Roman numeral conversion. Converting {query} gives {value}, so the query evaluates to {row['answer']}.</think>"


def build_gravity_trace(row: dict[str, str], payload: dict[str, Any]) -> str:
    query = row.get("query_raw", "").strip()
    g_value = payload.get("median_g", row.get("estimated_g", ""))
    return (
        f"<think>The examples fit d = 0.5 * g * t^2 with g ≈ {g_value}. "
        f"Substituting t = {query} gives {row['answer']}.</think>"
    )


def build_unit_trace(row: dict[str, str], payload: dict[str, Any]) -> str:
    query = row.get("query_raw", "").strip()
    ratio = payload.get("median_ratio", row.get("estimated_ratio", ""))
    return (
        f"<think>The examples show a fixed conversion ratio of {ratio}. "
        f"Applying that ratio to {query} gives {row['answer']}.</think>"
    )


def build_text_trace(row: dict[str, str], payload: dict[str, Any]) -> str:
    solver = row.get("teacher_solver_candidate", "")
    if solver == "text_word_dictionary":
        return (
            f"<think>The examples define a consistent word-level substitution. "
            f"Applying the same dictionary to the query yields {row['answer']}.</think>"
        )
    unknown_chars = row.get("text_unknown_chars", "")
    if unknown_chars:
        return (
            f"<think>The examples define a consistent substitution cipher, and the query characters {unknown_chars} are already covered. "
            f"Applying that mapping yields {row['answer']}.</think>"
        )
    return (
        f"<think>The examples define a consistent character substitution. "
        f"Applying the same mapping to the query yields {row['answer']}.</think>"
    )


def bit_rule_description(row: dict[str, str], payload: dict[str, Any]) -> str:
    solver = row.get("teacher_solver_candidate", "")
    if solver in {"binary_structured_byte_formula", "binary_structured_byte_formula_abstract"}:
        detail = row.get("bit_structured_formula_name", "") or row.get("bit_structured_formula_abstract_family", "")
        if detail:
            return humanize_rule_name(detail)
    if solver == "binary_byte_transform":
        names = row.get("bit_byte_transform_names", "") or "byte transform"
        return humanize_rule_name(names.split("|")[0])
    if solver == "binary_affine_xor":
        return "an affine XOR relation over the byte"
    if solver == "binary_three_bit_boolean":
        return "a verified 3-bit boolean rule"
    if solver == "binary_two_bit_boolean":
        return "a verified 2-bit boolean rule"
    if solver == "binary_bit_permutation_bijection":
        return "a fixed bit permutation/inversion"
    if solver == "binary_bit_permutation_independent":
        return "an independent bit permutation/inversion"
    detail = payload.get("structured_formula_name", "")
    return humanize_rule_name(str(detail)) if detail else humanize_rule_name(solver)


def build_bit_trace(row: dict[str, str], payload: dict[str, Any]) -> str:
    description = bit_rule_description(row, payload)
    return (
        f"<think>The examples match the verified bit rule {description}. "
        "I apply the same transformation to the query byte and keep the result as one exact 8-bit binary string with leading zeros. "
        "I will present only that final byte in the box.</think>"
    )


def build_symbol_trace(row: dict[str, str], payload: dict[str, Any]) -> str:
    operator = row.get("symbol_query_operator", "") or str(payload.get("query_operator", "")).strip()
    formula = row.get("symbol_numeric_formula_name", "") or str(payload.get("formula_name", "")).strip()
    if formula:
        formula_text = humanize_rule_name(formula)
        return (
            f"<think>For operator {operator}, the examples fit the verified rule {formula_text}. "
            f"Applying that rule to the query gives {row['answer']}.</think>"
        )
    return (
        f"<think>The examples support one verified symbol transformation for operator {operator}. "
        f"Applying it to the query gives {row['answer']}.</think>"
    )


def build_verified_cot(row: dict[str, str]) -> str:
    payload = parse_family_payload(row)
    family = row.get("family", "")
    if family == "roman_numeral":
        return build_roman_trace(row, payload)
    if family == "gravity_constant":
        return build_gravity_trace(row, payload)
    if family == "unit_conversion":
        return build_unit_trace(row, payload)
    if family == "text_decryption":
        return build_text_trace(row, payload)
    if family == "bit_manipulation":
        return build_bit_trace(row, payload)
    if family == "symbol_equation":
        return build_symbol_trace(row, payload)
    raise ValueError(f"Unsupported family for trace generation: {family}")


def build_output_rows(selected_verified: list[dict[str, str]], selected_answer_only: list[dict[str, str]]) -> list[dict[str, str]]:
    output_rows: list[dict[str, str]] = []
    for row in selected_verified:
        output_rows.append({
            "id": row["id"],
            "prompt": row["prompt"],
            "answer": row["answer"],
            "generated_cot": build_verified_cot(row),
            "label": FAMILY_LABELS[row["family"]],
            "assistant_style": "trace_boxed",
            "source_selection_tier": row["selection_tier"],
        })
    for row in selected_answer_only:
        output_rows.append({
            "id": row["id"],
            "prompt": row["prompt"],
            "answer": row["answer"],
            "generated_cot": "",
            "label": FAMILY_LABELS[row["family"]],
            "assistant_style": "boxed_only",
            "source_selection_tier": row["selection_tier"],
        })
    return output_rows


def validate_output(rows: list[dict[str, str]], expected_size: int) -> None:
    if len(rows) != expected_size:
        raise ValueError(f"Expected {expected_size} rows, found {len(rows)}")
    ids = [row["id"] for row in rows]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate ids detected in output rows")
    for row in rows:
        if set(row) != set(OUTPUT_COLUMNS):
            raise ValueError(f"Unexpected output columns for row {row.get('id', '')}")
        if row["assistant_style"] == "trace_boxed":
            if not row["generated_cot"].startswith("<think>") or not row["generated_cot"].endswith("</think>"):
                raise ValueError(f"Trace rows must be wrapped by <think>: {row['id']}")
            if row["label"] == "binary" and row["answer"] in row["generated_cot"]:
                raise ValueError(f"Binary trace should not repeat the final answer outside the box: {row['id']}")
        elif row["assistant_style"] == "boxed_only":
            if row["generated_cot"]:
                raise ValueError(f"boxed_only rows must not carry generated_cot: {row['id']}")
        else:
            raise ValueError(f"Unsupported assistant_style: {row['assistant_style']}")


def write_csv(path: str, rows: list[dict[str, str]], fieldnames: list[str]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def build_manifest(source_rows: list[dict[str, str]], output_rows: list[dict[str, str]]) -> dict[str, Any]:
    source_index = {row["id"]: row for row in source_rows}
    family_counts = Counter(source_index[row["id"]]["family"] for row in output_rows)
    source_tier_counts = Counter(row["source_selection_tier"] for row in output_rows)
    style_counts = Counter(row["assistant_style"] for row in output_rows)
    return {
        "subsample_size": len(output_rows),
        "family_counts": dict(sorted(family_counts.items())),
        "source_selection_tier_counts": dict(sorted(source_tier_counts.items())),
        "assistant_style_counts": dict(sorted(style_counts.items())),
        "training_plumbing": {
            "assistant_only_loss": True,
            "boxed_instruction": BOXED_INSTRUCTION,
            "max_seq_len": MAX_SEQ_LEN,
            "epochs": NUM_EPOCHS,
            "learning_rate": LR,
            "lora_rank": LORA_RANK,
            "target_modules_default": "all-linear",
        },
    }


def build_phase1_dataset(source_csv: str, output_csv: str, manifest_json: str) -> tuple[pl.DataFrame, dict[str, Any]]:
    source_rows = load_source_rows(source_csv)
    verified = verified_rows(source_rows)
    answer_only = answer_only_rows(source_rows)
    selected_verified = select_verified_mix(verified, seed=SEED)
    used_ids = {row["id"] for row in selected_verified}
    selected_answer_only = select_answer_only_mix(answer_only, used_ids=used_ids, seed=SEED)
    output_rows = build_output_rows(selected_verified, selected_answer_only)
    validate_output(output_rows, expected_size=SUBSAMPLE_SIZE)
    manifest = build_manifest(source_rows, output_rows)
    write_csv(output_csv, output_rows, OUTPUT_COLUMNS)
    with open(manifest_json, "w", encoding="utf-8") as fp:
        json.dump(manifest, fp, ensure_ascii=False, indent=2)
        fp.write("
")
    frame = pl.read_csv(output_csv, schema_overrides={
        "id": pl.Utf8,
        "prompt": pl.Utf8,
        "answer": pl.Utf8,
        "generated_cot": pl.Utf8,
        "label": pl.Utf8,
        "assistant_style": pl.Utf8,
        "source_selection_tier": pl.Utf8,
    })
    return frame, manifest


In [ ]:
if os.path.exists(PREBUILT_PHASE1_DATASET_CSV):
    print(f"Using prebuilt Phase 1 CSV: {PREBUILT_PHASE1_DATASET_CSV}")
    DATASET_CSV = PREBUILT_PHASE1_DATASET_CSV
    MANIFEST_JSON = None
else:
    print(f"Building Phase 1 CSV from source analysis: {SOURCE_ANALYSIS_CSV}")
    if not os.path.exists(SOURCE_ANALYSIS_CSV):
        raise FileNotFoundError(
            "Neither the prebuilt Phase 1 CSV nor the source analysis CSV is available. "
            "Attach one of them as a Kaggle dataset input."
        )
    DATASET_CSV = WORKING_DATASET_CSV
    MANIFEST_JSON = WORKING_MANIFEST_JSON
    train_df, phase1_manifest = build_phase1_dataset(SOURCE_ANALYSIS_CSV, DATASET_CSV, MANIFEST_JSON)
    print(json.dumps(phase1_manifest["family_counts"], ensure_ascii=False, sort_keys=True))
    print(json.dumps(phase1_manifest["assistant_style_counts"], ensure_ascii=False, sort_keys=True))

CSV_SCHEMA = {
    "id": pl.Utf8,
    "prompt": pl.Utf8,
    "answer": pl.Utf8,
    "generated_cot": pl.Utf8,
    "label": pl.Utf8,
    "assistant_style": pl.Utf8,
    "source_selection_tier": pl.Utf8,
}
train_df = pl.read_csv(DATASET_CSV, schema_overrides=CSV_SCHEMA)
if len(train_df) != SUBSAMPLE_SIZE:
    raise ValueError(f"Expected {SUBSAMPLE_SIZE} rows in {DATASET_CSV}, found {len(train_df)}")
print(train_df.group_by(["label", "assistant_style"]).len().sort(["label", "assistant_style"]))


### 💬 Assistant-Only Prompt Construction

- User message stays README-compatible
- Assistant completion is either:
  - `trace_boxed`: verified reasoning trace + boxed answer
  - `boxed_only`: final boxed answer only
- Labels are masked so only assistant tokens contribute to the loss

✅ This is the core Phase 1 change

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def apply_chat_template_safe(messages, *, add_generation_prompt, enable_thinking):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )
    except Exception:
        rendered = []
        for message in messages:
            rendered.append(f"<|{message['role']}|>\n{message['content']}")
        if add_generation_prompt:
            rendered.append("<|assistant|>\n<think>")
        return "\n".join(rendered)


def build_user_message(prompt: str) -> str:
    return prompt + "\n" + BOXED_INSTRUCTION


def render_assistant_message(example: dict[str, str]) -> str:
    if example["assistant_style"] == "trace_boxed":
        return f"{example['generated_cot']}\n\n\boxed{{{example['answer']}}}"
    if example["assistant_style"] == "boxed_only":
        return f"\boxed{{{example['answer']}}}"
    raise ValueError(f"Unsupported assistant_style: {example['assistant_style']}")


def tokenize_training_row(example: dict[str, str]) -> dict[str, Any]:
    user_message = build_user_message(example["prompt"])
    assistant_message = render_assistant_message(example)
    prompt_prefix = apply_chat_template_safe(
        [{"role": "user", "content": user_message}],
        add_generation_prompt=True,
        enable_thinking=True,
    )
    full_text = apply_chat_template_safe(
        [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message},
        ],
        add_generation_prompt=False,
        enable_thinking=True,
    )
    prompt_ids = tokenizer(prompt_prefix, add_special_tokens=False)["input_ids"]
    full_encoded = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=MAX_SEQ_LEN)
    input_ids = list(full_encoded["input_ids"])
    attention_mask = list(full_encoded["attention_mask"])
    if len(prompt_ids) > len(input_ids):
        raise ValueError(f"Prompt prefix exceeds truncated full sequence for row {example['id']}")
    if input_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError(f"Prompt prefix tokenization mismatch for row {example['id']}")
    labels = [-100] * len(prompt_ids) + input_ids[len(prompt_ids):]
    return {
        "id": example["id"],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

hf_source = Dataset.from_pandas(train_df.to_pandas())
train_dataset = hf_source.map(tokenize_training_row, remove_columns=hf_source.column_names)
print(train_dataset[0].keys())
print(f"Rows tokenized: {len(train_dataset)}")


### 🧠 Model Loading & Assistant-Only LoRA Training

- Keeps the baseline LoRA config (`rank=32`, `all-linear`, `dropout=0.05`)
- Replaces `SFTTrainer` with `Trainer`
- Uses a custom collator because labels are already masked

✅ Same baseline training spirit, but Phase 1 loss is assistant-only

In [ ]:
class AssistantOnlyDataCollator:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        max_len = max(len(feature["input_ids"]) for feature in features)
        input_ids_batch = []
        attention_batch = []
        labels_batch = []
        for feature in features:
            pad = max_len - len(feature["input_ids"])
            input_ids_batch.append(feature["input_ids"] + [self.pad_token_id] * pad)
            attention_batch.append(feature["attention_mask"] + [0] * pad)
            labels_batch.append(feature["labels"] + [-100] * pad)
        return {
            "input_ids": torch.tensor(input_ids_batch, dtype=torch.long),
            "attention_mask": torch.tensor(attention_batch, dtype=torch.long),
            "labels": torch.tensor(labels_batch, dtype=torch.long),
        }

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map={"": 0},
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
model.gradient_checkpointing_enable()

for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=5,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    data_collator=AssistantOnlyDataCollator(tokenizer.pad_token_id),
)

print("Starting Phase 1 assistant-only training...")
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")


### 📦 Create Submission ZIP

- Same final packaging step as the baseline notebook
- Verifies `adapter_config.json` and `adapter_model.safetensors`

✅ Ready for Kaggle submission

In [ ]:
zip_path = "/kaggle/working/submission.zip"
print(f"Packaging files from {OUTPUT_DIR}...")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

print(f"Created {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.1f} MB)")
with zipfile.ZipFile(zip_path, 'r') as zf:
    zip_contents = zf.namelist()
    print(f"Zip Contents: {zip_contents}")
    if "adapter_config.json" not in zip_contents:
        raise AssertionError("CRITICAL ERROR: adapter_config.json is missing from the zip. The Kaggle evaluation will fail.")
    if "adapter_model.safetensors" not in zip_contents:
        raise AssertionError("CRITICAL ERROR: adapter_model.safetensors is missing from the zip.")
print("✅ submission.zip is ready! You can now submit this file to the competition.")
